# BHaH Project Anatomy

Read the structure of a standalone BHaH project generated from the Cartesian wave equation.

Navigation: [Index](../index.ipynb) | Previous: [Multicoordinate Wave Project](../3-wave_equation/wave_equation_multicoordinates.ipynb) | Next: [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)


## Generate a Standalone BHaH Project
This is the same real Cartesian wave project, read as an infrastructure artifact rather than as a first-run exercise.

## Import Project Inspection Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.


In [ ]:
from pathlib import Path
import re, shutil, subprocess, sys, tempfile


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)


def require_toolchain():
    if shutil.which("make") is None:
        raise RuntimeError(
            "This notebook requires make to build the generated project."
        )
    if not any(shutil.which(name) for name in ["cc", "gcc", "clang"]):
        raise RuntimeError(
            "This notebook requires a C compiler such as cc, gcc, or clang."
        )


## Create an Anatomy Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [ ]:
PROJECT_NAME = "wave_equation_cartesian"
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_cartesian_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME


## Generate a Fresh Anatomy Project

This command invokes the same module a learner can run from a terminal and then verifies that the project directory exists.


In [ ]:
command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command: python -m nrpy.examples.wave_equation_cartesian")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print("project path: project/wave_equation_cartesian")


## Inventory and Core Files

In [ ]:
required = [
    "Makefile",
    "BHaH_function_prototypes.h",
    "commondata_struct_set_to_default.c",
    "params_struct_set_to_default.c",
    "rhs_eval.c",
    "diagnostics.c",
    "wave_equation_cartesian.par",
]
for relative_path in required:
    path = PROJECT_DIR / relative_path
    if not path.is_file():
        raise FileNotFoundError(path)
print("top-level project entries:")
for path in sorted(PROJECT_DIR.iterdir()):
    print(path.name + ("/" if path.is_dir() else ""))


## Step 4: Read the Parameter File

This is the runtime file a learner edits before running the executable.

In [ ]:
print(
    (PROJECT_DIR / "wave_equation_cartesian.par").read_text(
        encoding="utf-8", errors="replace"
    )
)


## Step 5: Read the Prototype Header

The header shows the callable generated functions.

In [ ]:
print(
    (PROJECT_DIR / "BHaH_function_prototypes.h").read_text(
        encoding="utf-8", errors="replace"
    )
)


## Step 6: Read Default-Setting Sources

These files initialize generated runtime structures.

In [ ]:
for relative_path in [
    "commondata_struct_set_to_default.c",
    "params_struct_set_to_default.c",
]:
    print(f"--- {relative_path} ---")
    print((PROJECT_DIR / relative_path).read_text(encoding="utf-8", errors="replace"))


## Step 7: Read the RHS Source

The RHS source contains the wave-equation update kernel.

In [ ]:
print((PROJECT_DIR / "rhs_eval.c").read_text(encoding="utf-8", errors="replace"))


## Step 8: Read the Diagnostics Source

Diagnostics write the errors used by convergence notebooks.

In [ ]:
print((PROJECT_DIR / "diagnostics.c").read_text(encoding="utf-8", errors="replace"))


The inspected files show how a BHaH project connects parameters, prototypes, generated C functions, and diagnostics. This structure is the target produced by the earlier C-function and wave notebooks.


## Continue to ETLegacy Infrastructure
- [C Function Registration](../1-intro/c_function.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
- [ETLegacy Core Infrastructure](etlegacy_core_infrastructure.ipynb)
